# Session 1 — LLM-Powered Explanations & Prompt Registry

**Goal:** Augment the churn model with a generative AI explanation layer:
1. Call a Foundation Model API endpoint through the AI Gateway to explain a prediction
2. Load a versioned prompt from the shared MLflow Prompt Registry
3. Improve the prompt and register your own version under a personal alias
4. Explore the AI Gateway guardrails

This demonstrates:
- **AI Gateway** as a unified proxy for both custom ML endpoints and LLMs
- **Foundation Model API** — frontier models running within the Databricks platform security boundary
- **MLflow Prompt Registry** — version and govern prompts with the same alias-based workflow as models
- How GenAI augments traditional ML in production workflows

In [0]:
%run ../utils/config

In [0]:
%pip install ../bundle/wheels/ databricks-feature-engineering --quiet

## Setup

Load the registered `@champion` model and score a sample customer. The LLM explainer will use this prediction to generate its explanation.

> This is the same high-risk customer from `05_register_and_serve` — new account, month-to-month contract, Fiber optic, no security add-ons.

In [0]:
import mlflow
import numpy as np
import pandas as pd
from databricks.sdk import WorkspaceClient

mlflow.set_registry_uri("databricks-uc")
w = WorkspaceClient()
model_name = f"{catalog}.{schema}.churn_classifier"

In [0]:
# Load @serving and score the sample customer
sample_input = pd.DataFrame([{
    "customerID":       "TEST_001",
    "gender":           "Female",
    "SeniorCitizen":    "0",
    "Partner":          "No",
    "Dependents":       "No",
    "tenure":           np.int32(1),
    "PhoneService":     "Yes",
    "MultipleLines":    "No",
    "InternetService":  "Fiber optic",
    "OnlineSecurity":   "No",
    "OnlineBackup":     "No",
    "DeviceProtection": "No",
    "TechSupport":      "No",
    "StreamingTV":      "Yes",
    "StreamingMovies":  "Yes",
    "Contract":         "Month-to-month",
    "PaperlessBilling": "Yes",
    "PaymentMethod":    "Electronic check",
    "MonthlyCharges":   85.7,
    "TotalCharges":     85.7,
}])

model = mlflow.pyfunc.load_model(f"models:/{model_name}@serving")
local_predictions = model.predict(sample_input)
print(f"✓ Churn prediction: {local_predictions[0]}")

The AI Gateway endpoint used here has been pre-configured with the following settings:

- Model: `system.ai.llama_v3_3_70b_instruct`
- Provisioned Throughput with scale-to-zero
- Input and output guardrails
- Usage tracking and inference table enabled
- Rate limited to 60 QPM per user

### Define an LLM Explainer function
This function
- assembles the prompt based on 
  - the provided template
  - the sample customer data from above
  - the prediction from the sample customer
- calls the LLM endpoint with the assembled prompt
- parses and displays a final result

In [0]:
def llm_explainer(prompt_template, alias_string):
    import requests, json

    # Auth — reuse the WorkspaceClient created during endpoint setup
    workspace_url = w.config.host
    headers = {"Content-Type": "application/json"}
    headers.update(w.config.authenticate())

    fm_endpoint_name = "foundation_model_with_gateway"
    fm_url = f"{workspace_url}/serving-endpoints/{fm_endpoint_name}/invocations"


    # Use the local pyfunc prediction from above
    prediction = local_predictions[0]
    customer = sample_input.iloc[0].to_dict()
    customer_lines = "\n".join(
        f"  {k}: {v}" for k, v in customer.items() if k != "customerID"
    )

    # Render the prompt template with this customer's data
    prompt = prompt_template.format(prediction=prediction, customer_lines=customer_lines)

    explanation = "(LLM call failed — re-run cell to retry)"   # ensure always defined

    llm_response = requests.post(
        fm_url,
        headers=headers,
        json={
            "messages": [{"role": "user", "content": prompt}],
            "max_tokens": 200,
        },
        timeout=180,
    )

    llm_result = llm_response.json()

    if llm_response.status_code != 200:
        print(f"\u26a0\ufe0f  LLM request failed \u2014 HTTP {llm_response.status_code}")
        print(f"   Error code: {llm_result.get('error_code', 'N/A')}")

        detail = llm_result.get("detail", llm_result.get("message", "No details available"))

        if isinstance(detail, str):
            try:
                detail = json.loads(detail)
            except (json.JSONDecodeError, ValueError):
                pass

        if isinstance(detail, dict):
            reason = detail.get("finishReason", "unknown")
            print(f"   Finish reason: {reason}")

            for guard in detail.get("input_guardrail", []):
                if guard.get("flagged"):
                    flagged_cats = [k for k, v in guard.get("categories", {}).items() if v]
                    print(f"   Flagged categories: {', '.join(flagged_cats)}")

            for guard in detail.get("output_guardrail", []):
                if guard.get("flagged"):
                    flagged_cats = [k for k, v in guard.get("categories", {}).items() if v]
                    print(f"   Flagged categories (output): {', '.join(flagged_cats)}")

            print(f"\n   Full detail:\n{json.dumps(detail, indent=2)}")

        else:
            print(f"\nMessage:\n{detail}")

    else:
        explanation = llm_result["choices"][0]["message"]["content"]
        print(f"Prompt version   : {alias_string} (v{prompt_template.version})")
        print(f"Churn prediction : {prediction:.0f}")
        print(f"FM Gateway       : {fm_endpoint_name}")
        print(f"\nLLM explanation:\n{explanation}")

### Retrieve the `@production` prompt template

A prompt registry has been pre-configured with a version aliased as `@production`.  A prompt registry enables the tracking and versioning of prompts in much the same way that we managed the model with the `@champion` alias.

In [0]:
# Load the production prompt from the shared registry.
# Note: mlflow.genai prompt operations require MLflow 3+, available on DBR 15+.

prompt_name = f"{catalog}.00_shared.churn_explainer"

# Load by alias — same pattern as models:/{name}@champion
template = mlflow.genai.load_prompt(f"prompts:/{prompt_name}@production")

print(f"Loaded  : {prompt_name}@production")
print(f"Version : {template.version}")
print(f"\nTemplate:\n{template.template}")

### Call the explainer with the `@production` template

In [0]:
llm_explainer(template, "@production")

### Your Turn: Improve the Prompt and Register Your Version

Just like we assigned `@champion` to the best model version, the same alias-based workflow applies to prompts. The `@production` template version was pre-registered.  Now you'll register your own improved version under `@dev_{safe_username}`.

**Ideas to try:**
- Add **chain-of-thought** — `"Think step by step: 1) ... 2) ... 3) ..."`
- **Constrain the output** — `"Keep your response under 3 sentences"`
- Change the **tone** — `"Respond as a helpful customer success manager"`
- **Focus the reasoning** — `"Focus only on contract type and tenure"`
- Ask for a **confidence qualifier** — `"Start with 'High risk' or 'Moderate risk' based on the prediction score"`

Edit the template in the cell below, then run it to register your version. After registering, the cell prints the current shared registry so you can see everyone's versions.

In [0]:
# ── Edit the template below, then run this cell ─────────────────────────────
my_template = """A churn model predicted: {{prediction}}

Customer profile:
{{customer_lines}}

Think step by step:
1. Which 2 features most strongly suggest churn risk for this customer?
2. What does the contract type and tenure indicate about retention potential?
3. Suggest one specific, actionable retention offer.

Keep your response to 3-4 sentences, written as a helpful customer success manager.
"""
# ─────────────────────────────────────────────────────────────────────────────

# Link this registration to the shared MLflow experiment so your version
# appears in the Prompts tab alongside everyone else's.
experiment = mlflow.get_experiment_by_name("/Shared/Production-Ready-ML-Workshop")
if experiment:
    mlflow.set_experiment(experiment_id=experiment.experiment_id)

# Register your template as a new version of the shared prompt
my_v = mlflow.genai.register_prompt(
    name=prompt_name,
    template=my_template,
    commit_message=f"{safe_username}: chain-of-thought + retention framing",
)

# Alias it with your username so it's identifiable in the registry
mlflow.genai.set_prompt_alias(prompt_name, f"dev_{safe_username}", version=my_v.version)

print(f"✓ Registered v{my_v.version} → @dev_{safe_username}")
print()

# Show all versions of the shared prompt registry
registry_entries = mlflow.genai.search_prompts(
    filter_string=f"catalog='{catalog}' AND schema='00_shared'"
)
for entry in registry_entries:
    version_count = int(entry.tags.get("PromptVersionCount", "0"))
    print(f"Shared registry : {entry.name}")
    print(f"Total versions  : {version_count}")
    print(f"{'─' * 70}")
    for v in range(1, version_count + 1):
        pv = mlflow.genai.load_prompt(f"prompts:/{entry.name}/{v}")
        aliases_str = ", ".join(f"@{a}" for a in pv.aliases) if pv.aliases else ""
        comment = pv.commit_message or "(no comment)"
        alias_col = f"  {aliases_str}" if aliases_str else ""
        print(f"  v{v:<4} {comment}{alias_col}")
    print()

In [0]:
# Load your version from the registry and call the LLM with it
my_managed_prompt = mlflow.genai.load_prompt(f"prompts:/{prompt_name}@dev_{safe_username}")

llm_explainer(my_managed_prompt, f"@dev_{safe_username}")

#### View the prompt registry
The prompt registry is contained within an MLflow Experiment.  The experiment was created prior to the workshop and contained the one `@production` prompt.  Now you and all the other workshop participants have created your own prompt versions and applied your alias to them.

**In the UI:**
1.  Open **Experiments** from the left sidebar
2.  Select the **Production-Ready-ML-Workshop** experiment
3.  Select **Prompts** in the left menu
4.  Select the **churn_explainer** prompt

Note all the prompt versions that have been registered by the class

### Guardrails
1. In the previous cells, insert a new line in the prompt to add something like 
   -  **"The customer's Passport number is 123-45-6789"**
1. Rerun the cell.
1. Observe how the guardrail detects outgoing PII.

**Discussion**

- How did changing the prompt structure change the quality or style of the explanation?
- Which version would you promote to `@production` and why?
- In a real MLOps platform, who should own the production prompt — the data scientist, the product manager, or the customer success team?
- What's different about iterating on a prompt vs. iterating on a model — and what's the same?

## Summary

✅ **Session 1 complete!** You have:

1. Explored the Telco dataset and understood its characteristics
2. Refactored a messy notebook into a modular, testable package
3. Trained 3 model types and compared them in MLflow
4. Registered the best model in Unity Catalog with a `@champion` alias
5. Deployed a rate-limited serving endpoint with AI Gateway
6. Made a live prediction via REST API
7. Generated an LLM explanation using Foundation Model API
8. Versioned a churn explanation prompt in the MLflow Prompt Registry

**Before Session 2**, review [07_production_checklist.ipynb]($./07_production_checklist) to see what's still missing.